## Build a chatbot using LangGraph that remembers the conversation within a session.

### What it should do:

* User types a message
* Agent responds using an LLM
* User types another message — agent remembers the previous one
* Prove memory works by asking "what did I say earlier?"

### Constraints:

* Use MemorySaver as checkpointer
* Use MessagesState for state
* Must use a thread ID in the config when invoking
* No LangChain chains — raw LangGraph only

In [1]:
# python
 
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

True

In [2]:
print("GROQ_API_KEY available:", bool(os.getenv("GROQ_API_KEY")))

GROQ_API_KEY available: True


In [3]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
import uuid # for thread_id

In [4]:
model = init_chat_model(
    model="groq/compound-mini",       # The specific Groq model ID
    model_provider="groq",        # Specifies the provider
    temperature=0                 # Optional parameters
)

In [5]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def chat(state: State) -> str:
    response = model.invoke(state["messages"])
    return {"messages": [response]}

In [6]:
# workflow
builder = StateGraph(State)
builder.add_node("model", chat)
builder.add_edge(START, "model")
builder.add_edge("model", END)

In [7]:
checkpoint = InMemorySaver()
store = InMemoryStore()

graph =builder.compile(checkpointer=checkpoint, store=store)

In [8]:
def chat_model(user_input: str, thread_id: str = None) -> str:
    config = {"configurable": {"thread_id": thread_id or str(uuid.uuid4())[:255]}}


    result = graph.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config,
    )
    return result["messages"][-1].content

In [9]:
def run_chat_loop():
    thread_id = str(uuid.uuid4())  # generated ONCE here
    print(f"Session started. Thread ID: {thread_id}")
    print("Welcome to the chat! Type 'exit' to quit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            print("Exiting the chat. Goodbye!")
            break
        response = chat_model(user_input, thread_id)
        print(f"AI: {response}")

In [ ]:
run_chat_loop()

Session started. Thread ID: 6f7c09fb-525f-4fa7-8f7a-c7b83956c5fb
Welcome to the chat! Type 'exit' to quit.
AI: Hello! How can I help you today?
AI: Hey there! Let me know if there's anything you'd like to talk about or ask.
